# ValidationDataSet

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `data`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/ValidationDataSet.md`


Notebook source link: [ValidationDataSet.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/ValidationDataSet.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "ValidationDataSet"
FAMILY = "data"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")


In [ ]:
# Data-style workflow: trial-to-trial variability and PSTH-like estimates.
dt = 0.001
time = np.arange(0.0, 1.2, dt)
n_trials = 30

rate = 5.0 + 8.0 * (time > 0.35) + 4.0 * np.sin(2.0 * np.pi * 2.0 * time)
rate = np.clip(rate, 0.2, None)

trial_matrix = np.zeros((n_trials, time.size), dtype=float)
for k in range(n_trials):
    jitter = 0.6 + 0.8 * rng.random()
    p = np.clip(rate * jitter * dt, 0.0, 0.6)
    trial_matrix[k, :] = rng.binomial(1, p)

psth = trial_matrix.mean(axis=0) / dt
sem = trial_matrix.std(axis=0, ddof=1) / np.sqrt(n_trials) / dt

rates, prob_mat, sig_mat = DecodingAlgorithms.compute_spike_rate_cis(trial_matrix)

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=False)
for k in range(min(18, n_trials)):
    t_spk = time[trial_matrix[k] > 0]
    axes[0].vlines(t_spk, k + 0.6, k + 1.4, linewidth=0.5)
axes[0].set_title(f"{TOPIC}: trial raster")
axes[0].set_ylabel("trial")

axes[1].plot(time, psth, color="tab:blue", linewidth=1.2)
axes[1].fill_between(time, psth - sem, psth + sem, color="tab:blue", alpha=0.2)
axes[1].set_ylabel("Hz")
axes[1].set_title("PSTH mean +/- SEM")

im = axes[2].imshow(prob_mat, aspect="auto", origin="lower", cmap="viridis")
axes[2].set_title("Trial-by-trial spike-rate p-values")
axes[2].set_xlabel("trial")
axes[2].set_ylabel("trial")
fig.colorbar(im, ax=axes[2], fraction=0.03, pad=0.02)

plt.tight_layout()
plt.show()

print("significant pair count", int(sig_mat.sum()))
assert np.allclose(prob_mat, prob_mat.T, atol=1e-12)
assert np.all(np.diag(prob_mat) == 1.0)


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
